# 00 · 先确认数据与损失没有算错

这本 notebook 是**基础设施自检与机制导览**。它不下载模型、不训练。请在 Mac 或 Colab 的 CPU runtime 运行；分配 GPU runtime 后，即使代码只用 CPU，也可能消耗额度；输出来自合成样本和小数组，不能说明 DFT 的模型效果。

我们先检查三个容易悄悄改变实验含义的地方：哪些 token 被学习、模型在预测哪一个 token、累计梯度时怎样归一化。接着才有资格做一次真实模型 smoke run。

**已执行状态：** 此交付保存了实际 CPU 输出。真实 PyTorch 梯度测试是否执行，请以末尾测试结果为准；没有 GPU 训练结果。

## 1. 找到完整实验包

从包的根目录或 `notebooks/` 启动即可。需要 Python、NumPy 和 Jupyter 内核。这里不安装依赖，也不读取个人账号。

Local Mac validation on 2026-10-15: this copy was executed top-to-bottom in a real Jupyter kernel using the existing .venv-research environment. Dependency installation and training remained disabled. The two PyTorch checks were skipped because torch is absent; 15 other foundation checks passed. No model was downloaded, no GPU training ran, and no research result was produced. The original cloud notebook is preserved in the downloaded package.

In [1]:
from pathlib import Path
import sys

# 完整解压基础包；只上传 ipynb 会缺少 experiments/ 和 tests/。
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/PostTraining'),
              Path.cwd() / 'PostTraining']
ROOT = next((p for p in candidates if (p/'experiments/contracts.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('先把完整 PostTraining 文件夹解压至 /content，或将工作目录切到包根目录。')
sys.path.insert(0, str(ROOT))
print('已找到实验包:', ROOT.name)


import json
from experiments.preflight import environment_report
report = environment_report()
print(json.dumps({k: report[k] for k in ('python','packages','cuda_available','gpu')}, ensure_ascii=False, indent=2))

已找到实验包: PostTraining
{
  "python": "3.12.13",
  "packages": {
    "numpy": "2.5.4",
    "torch": null,
    "transformers": null,
    "peft": null,
    "accelerate": null,
    "huggingface-hub": null,
    "tokenizers": null,
    "safetensors": null
  },
  "cuda_available": false,
  "gpu": null
}


## 2. 看一条明确标注的合成样本

内置 12 条训练样本、4 条验证样本只是手写加法题。它们用于检查文件读取、mask、保存与生成是否通畅，不是 GSM8K、不是研究语料，也没有测试集。验证集不进入训练梯度。

以后换成本地 JSONL，每行保留 `id`、`prompt`、`completion` 三个非空字符串；还需另外记录来源、许可和划分方法。

## 3. 只监督回答，并正确移动一位

Base 模型使用固定文本前缀 `Question: ...\nAnswer:`。训练和生成调用同一个 prompt 编码。回答单独编码后追加一个 EOS；prompt 的标签是 `-100`，不计入损失。

下面的字符 tokenizer **只是测试替身，不是 Qwen tokenizer**。它把一个字符映射成一个整数，让边界可以直接检查。真实模型会使用自己的 tokenizer。

超出 `max_length` 的样本整条剔除并记录，避免截掉最终答案却仍称它为原任务。本包暂不做 packing 或多样本 padding。

In [2]:
from experiments.contracts import load_jsonl, ensure_disjoint, canonical_hash
train_rows = load_jsonl(ROOT/'experiments/data/smoke_train.jsonl')
val_rows = load_jsonl(ROOT/'experiments/data/smoke_val.jsonl')
ensure_disjoint(train_rows, val_rows)
print(json.dumps({'train_rows':len(train_rows), 'validation_rows':len(val_rows),
                 'train_sha256':canonical_hash(train_rows), 'example':train_rows[0]}, ensure_ascii=False, indent=2))

from experiments.contracts import encode_record
from tests.test_contracts import CharacterTokenizer
item, issue = encode_record(train_rows[0], CharacterTokenizer())
prompt_length = len(item['prompt_ids'])
print({'prompt_tokens_masked':prompt_length, 'response_plus_eos_tokens':item['target_tokens'],
       'labels_at_boundary':item['labels'][prompt_length-2:prompt_length+4],
       'final_label_is_eos':item['labels'][-1] == 0})
assert all(x == -100 for x in item['labels'][:prompt_length])
assert item['labels'][prompt_length:] == item['input_ids'][prompt_length:]
# logits[t] 对 labels[t+1]。最后一个 logits 没有后继标签。
assert sum(x != -100 for x in item['labels'][1:]) == item['target_tokens']
print('mask 与 shift 一致。')

{
  "train_rows": 12,
  "validation_rows": 4,
  "train_sha256": "71a61677069188f23db76342e80345ddb3f75e25657d1dd0494a670f6758826f",
  "example": {
    "id": "synthetic-000",
    "prompt": "What is 2 + 3?",
    "completion": "2 + 3 = 5."
  }
}
{'prompt_tokens_masked': 32, 'response_plus_eos_tokens': 12, 'labels_at_boundary': [-100, -100, 33, 51, 33, 44], 'final_label_is_eos': True}
mask 与 shift 一致。


## 4. 比较两个损失的含义

对有效回答 token，SFT 使用 `-log p(target)`；这里的 DFT 对同一项乘 `p(target).detach()`，再按**有效 token 数量**取平均。`detach` 表示权重不反向求导。

两者可以共用 unweighted NLL 观察拟合变化；它们自己的优化损失不在同一个标度上。DFT 标量更小，不能据此说它训练得更好。这里也没有声称复现论文完整配置。

In [3]:
import numpy as np
from experiments.contracts import numpy_causal_loss
# 两个目标token的当前概率分别为0.2和0.8。
logits = np.log(np.array([[[0.2,0.8],[0.2,0.8],[0.5,0.5]]]))
labels = np.array([[-100,0,1]])
reference_results = {}
for objective in ('sft','dft'):
    result = numpy_causal_loss(logits, labels, objective)
    reference_results[objective] = result
    print(objective, {k: round(v,6) if isinstance(v,float) else v
                      for k,v in result.items() if k != 'gradient'})
print('DFT/SFT梯度倍率:',
      reference_results['dft']['gradient'][0,:2,0] / reference_results['sft']['gradient'][0,:2,0])

sft {'objective': 'sft', 'loss': 0.916291, 'unweighted_nll': 0.916291, 'mean_target_probability': 0.5, 'target_tokens': 2}
dft {'objective': 'dft', 'loss': 0.250201, 'unweighted_nll': 0.916291, 'mean_target_probability': 0.5, 'target_tokens': 2}
DFT/SFT梯度倍率: [0.2 0.8]


较不确定的第一个目标 token 在 DFT 中得到较小的梯度权重。这个小数组展示了计算规则；“什么时候这种权重有益”仍需要具体任务和模型证据。

## 5. 执行边界测试

覆盖：prompt mask、EOS、一次 shift、无有效标签、错误形状、超长边界、重复样本、训练/验证重叠、固定抽样顺序、不同长度样本的 token 总数归一化，以及 DFT 停止权重梯度。

如果环境没有 torch，两个真实 autograd 检查会显示 `skipped`，不会伪装成通过。GPU 集成另需在下一本 notebook 验证。

In [4]:
import unittest
suite = unittest.defaultTestLoader.discover(str(ROOT/'tests'), pattern='test_*.py')
result = unittest.TextTestRunner(verbosity=1).run(suite)
assert result.wasSuccessful(), '自检失败，先停止训练并修复。'
print({'tests_run':result.testsRun, 'passed':result.testsRun-len(result.skipped)-len(result.failures)-len(result.errors),
       'skipped':len(result.skipped), 'failures':len(result.failures), 'errors':len(result.errors)})

.

.

.

.

.

...

.

.

.

----------------------------------------------------------------------
Ran 17 tests in 0.248s

OK (skipped=2)


{'tests_run': 17, 'passed': 15, 'skipped': 2, 'failures': 0, 'errors': 0}


## 下一步

打开 `01_colab_sft_dft.ipynb`，先记录分配到的 GPU，再决定是否开启一次最多 **每分支 20 个 optimizer steps** 的 smoke run。首次目的是验证真实 tokenizer、CUDA、LoRA、日志与结果保存能协作；不能据此选论文题目或下“DFT 优于 SFT”的结论。

官方依据：[模型卡](https://huggingface.co/Qwen/Qwen3-0.6B-Base)、[PEFT LoRA](https://huggingface.co/docs/peft/v0.15.0/en/package_reference/lora)、[TRL 的 SFT/DFT 说明](https://huggingface.co/docs/trl/en/sft_trainer)。本包直接写出 loss，不依赖 TRL。